# RPG ASCII
Execute a célula, clique no desenho. O jogo começa automaticamente. Use as setas ou botões.

Python exibe HTML; JavaScript controla o jogo no navegador. Não precisa instalar pacotes.

A cobrinha atravessa as bordas e o próprio corpo. O Tetris usa blocos ASCII, o Pong é contra o computador e o RPG pede uma chave para abrir a porta.

In [ ]:
from IPython.display import Javascript, display

# Execute esta célula inteira. Clique no desenho e use as setas.
# Python exibe a saída; JavaScript controla o jogo no navegador.
display(Javascript(r"""(() => {
// O painel fica na saída desta célula, sem iframe dentro de iframe.
const painel = document.createElement('div');
painel.style = 'background:#111;color:#a6ff99;padding:12px;max-width:100%;font:14px monospace';
painel.innerHTML = "<b>RPG ASCII</b><p><button data-action=\"restart\">Reiniciar</button> <button data-action=\"pause\">Pausar / continuar</button></p><span role=\"status\"></span><pre tabindex=\"0\" aria-label=\"Jogo ASCII: use as setas\" style=\"font:14px/1.2 monospace;overflow:auto;outline:1px solid #486;padding:6px\"></pre><div><button data-key=\"ArrowUp\">Cima</button><button data-key=\"ArrowDown\">Baixo</button><button data-key=\"ArrowLeft\">Esquerda</button><button data-key=\"ArrowRight\">Direita</button></div>";
document.body.appendChild(painel);
const tela = painel.querySelector('pre'), aviso = painel.querySelector('[role="status"]');
let tempo = null, rodando = false, terminou = false;
function moldura(linhas) {
    const borda = '+' + '-'.repeat(linhas[0].length) + '+';
    return [borda, ...linhas.map(l => '|' + l + '|'), borda].join(String.fromCharCode(10));
}
// @ é o herói, K é a chave, D é a porta e # é uma parede.
const mapaInicial = [
    '################',
    '#@.....#.......#',
    '#.###..#..###..#',
    '#...#.....#K...#',
    '###.#..#..###..#',
    '#......#.......#',
    '#..########.#..#',
    '#...........#D.#',
    '################'
];
const largura = mapaInicial[0].length, altura = mapaInicial.length;
const teclas = {ArrowUp:[0,-1], ArrowDown:[0,1], ArrowLeft:[-1,0], ArrowRight:[1,0]};
let mapa, x, y, chave, passos, mensagem;
function reiniciar() {
    mapa = mapaInicial.map(linha => linha.split('')); x = 1; y = 1;
    mapa[y][x] = '.'; chave = false; passos = 0; mensagem = 'Encontre K e depois alcance D.';
}
function comando(tecla) {
    const direcao = teclas[tecla]; if (!direcao) return;
    const nx = x+direcao[0], ny = y+direcao[1];
    const destino = mapa[ny]?.[nx];
    if (!destino || destino === '#') { mensagem = 'Parede! Procure outro caminho.'; return; }
    if (destino === 'D' && !chave) { mensagem = 'A porta precisa da chave K.'; return; }
    x = nx; y = ny; passos++;
    if (destino === 'K') { chave = true; mapa[y][x] = '.'; mensagem = 'Chave encontrada! Procure a porta D.'; }
    if (destino === 'D') { terminou = true; mensagem = 'VOCE VENCEU! O tesouro e seu.'; }
}
function atualizar() {} // Este jogo avança a cada tecla, sem movimento automático.
function desenhar() {
    const desenho = mapa.map(linha => [...linha]); desenho[y][x] = '@';
    return desenho.map(linha => linha.join('')).join('\n') + '\nChave: ' + (chave ? 'sim' : 'nao') + ' | Passos: ' + passos + '\n' + mensagem + '\nSetas: caminhar | # parede | K chave | D saida';
}

function parar() { clearInterval(tempo); tempo = null; rodando = false; }
function mostrar() {
    tela.textContent = desenhar();
    if (terminou) { parar(); aviso.textContent = 'Fim da partida. Use Reiniciar.'; }
}
function continuar() {
    if (terminou) return;
    parar(); rodando = true;
    aviso.textContent = 'Clique no desenho e use as setas, ou toque nos botoes.';
    
    tela.focus();
}
function iniciar() { parar(); terminou = false; reiniciar(); mostrar(); continuar(); }
function receber(tecla) { if (rodando && !terminou) { comando(tecla); mostrar(); } }
tela.onkeydown = e => {
    if (['ArrowUp','ArrowDown','ArrowLeft','ArrowRight',' '].includes(e.key)) {
        e.preventDefault(); receber(e.key);
    }
};
painel.querySelectorAll('[data-key]').forEach(b => b.onclick = () => { receber(b.dataset.key); tela.focus(); });
painel.querySelector('[data-action="restart"]').onclick = iniciar;
painel.querySelector('[data-action="pause"]').onclick = () => {
    if (rodando) { parar(); aviso.textContent = 'Pausado.'; } else continuar();
};
window.addEventListener('pagehide', parar, {once:true});
iniciar();
})();"""))